In [ ]:
### PACKAGES ###

import torch
import torchvision.models as models
import torch.nn as nn
from torchvision.datasets import ImageFolder
import torch.optim as optim

import os
import numpy as np
import pandas as pd
import time 
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
### INPUTS ###
train_dir = ""
val_dir = ""
test_dir = ""

output_folder = ""
os.makedirs(output_folder, exist_ok=True)

In [ ]:
### DEVICE SETUP ### 

save = True
filename = "pytorch_environment_report.txt"
file_path = os.path.join(output_folder, filename)

print("="*40)
print("PyTorch Environment Report")
print("="*40)

print(f"PyTorch version:        {torch.__version__}")

# Check MPS (Metal) availability
mps_available = torch.backends.mps.is_available()
print(f"GPU available (MPS):    {mps_available}")

if mps_available:
    # Apple does not provide detailed GPU properties in PyTorch yet
    print(f"GPU device:             Apple GPU (Metal)")
    print(f"Current device:         mps")

# Set device
device = torch.device("mps") if mps_available else torch.device("cpu")
print("="*40)
print(f"Using device:           {device}")

if save:
    with open(file_path, "w") as f:
        def print_to_file(*args, **kwargs):
            print(*args, **kwargs, file=f)

        print_to_file("="*40)
        print_to_file("PyTorch Environment Report")
        print_to_file("="*40)

        print_to_file(f"PyTorch version:        {torch.__version__}")

        # Check MPS (Metal) availability
        mps_available = torch.backends.mps.is_available()
        print_to_file(f"GPU available (MPS):    {mps_available}")

        if mps_available:
            # Apple does not provide detailed GPU properties in PyTorch yet
            print_to_file(f"GPU device:             Apple GPU (Metal)")
            print_to_file(f"Current device:         mps")
        print_to_file("="*40)    
        print_to_file(f"Using device:           {device}")
                    
    print("Environment report saved to", filename)

In [2]:
# Check available models and their weights

for name in models.list_models():
    weights = models.get_model_weights(name)
    print(name, "→", [w.name for w in weights])

alexnet → ['IMAGENET1K_V1']
convnext_base → ['IMAGENET1K_V1']
convnext_large → ['IMAGENET1K_V1']
convnext_small → ['IMAGENET1K_V1']
convnext_tiny → ['IMAGENET1K_V1']
deeplabv3_mobilenet_v3_large → ['COCO_WITH_VOC_LABELS_V1']
deeplabv3_resnet101 → ['COCO_WITH_VOC_LABELS_V1']
deeplabv3_resnet50 → ['COCO_WITH_VOC_LABELS_V1']
densenet121 → ['IMAGENET1K_V1']
densenet161 → ['IMAGENET1K_V1']
densenet169 → ['IMAGENET1K_V1']
densenet201 → ['IMAGENET1K_V1']
efficientnet_b0 → ['IMAGENET1K_V1']
efficientnet_b1 → ['IMAGENET1K_V1', 'IMAGENET1K_V2']
efficientnet_b2 → ['IMAGENET1K_V1']
efficientnet_b3 → ['IMAGENET1K_V1']
efficientnet_b4 → ['IMAGENET1K_V1']
efficientnet_b5 → ['IMAGENET1K_V1']
efficientnet_b6 → ['IMAGENET1K_V1']
efficientnet_b7 → ['IMAGENET1K_V1']
efficientnet_v2_l → ['IMAGENET1K_V1']
efficientnet_v2_m → ['IMAGENET1K_V1']
efficientnet_v2_s → ['IMAGENET1K_V1']
fasterrcnn_mobilenet_v3_large_320_fpn → ['COCO_V1']
fasterrcnn_mobilenet_v3_large_fpn → ['COCO_V1']
fasterrcnn_resnet50_fpn → ['C

In [ ]:
# USE vit_b_16 trained on ImageNet ['IMAGENET1K_V1'] 
# loading the model with pretrained weights

weights = models.ViT_B_16_Weights.IMAGENET1K_V1
base_model = models.vit_b_16(weights=weights)

# preprocessing function for the model 
# input images need to be preprocessed in the same way as the training data of the pretrained model
preprocess = weights.transforms()


In [ ]:
# Replace the head of the model so it learns to classify our 17 classes instead of the 1000 ImageNet classes

num_classes = 17 # number of classes in dataset 

in_features = base_model.heads.head.in_features
base_model.heads.head = torch.nn.Linear(in_features, num_classes)

# print original model architecture
#print(base_model)

In [ ]:
## load previously trained model
model_path = ""
base_model = models.vit_b_16(weights=None)

# image preprocessing —> need to match training preprocessing
vit_weights = models.ViT_B_16_Weights.IMAGENET1K_V1
preprocess = vit_weights.transforms()

# modify the classifier head to match the number of classes (uses classes from test dataset)
### load classes names from training set ### 
train_dataset = ImageFolder(train_dir, transform=preprocess)
class_names = train_dataset.classes  # 17 jellyfish species
idx_to_class = {i: name for i, name in enumerate(class_names)}
num_classes = len(class_names)
    
# modify the head
base_model.heads.head = nn.Linear(base_model.heads.head.in_features, num_classes)

# load the trained weights
state = torch.load(model_path, map_location=device)
base_model.load_state_dict(state)

In [ ]:
### DATASET PREPARATION ###

train = ImageFolder(train_dir, transform=preprocess)
train_loader = torch.utils.data.DataLoader(train, batch_size=32, shuffle=True)

val = ImageFolder(val_dir, transform=preprocess)
val_loader = torch.utils.data.DataLoader(val, batch_size=128, shuffle=False)

test = ImageFolder(test_dir, transform=preprocess)
test_loader = torch.utils.data.DataLoader(test, batch_size=128, shuffle=False)


In [ ]:
### Check on data format ###

def jelly_labels(dataset):

    fig, axs = plt.subplots(3, 6, figsize=(16, 8))
    for ax in axs.ravel():
        # Pick random image
        idx = np.random.randint(0, len(dataset))
        image, label = dataset[idx]

        # Convert from tensor (C,H,W) -> (H,W,C)
        image_np = image.permute(1, 2, 0).numpy()

        # Show image
        ax.imshow(image_np, cmap = 'gray')

        # Show class name + numeric label
        class_name = dataset.classes[label]
        ax.set_title(f"{class_name} ({label})", fontsize=12)
        ax.axis("off")
        
    plt.tight_layout()
    plt.show()

jelly_labels(train)

In [ ]:
# Freeze all parameters first
for param in base_model.parameters():
    param.requires_grad = False

# Unfreeze the head
for param in base_model.heads.parameters():
    param.requires_grad = True

In [ ]:
### DEFINE LOSS FUNCTION AND OPTIMIZER ###

classifier = base_model

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, classifier.parameters()),
    lr=3e-4,  # typical LR for head
    weight_decay=1e-4
)

In [ ]:
### MANUAL TRAINING LOOP ###
classifier = base_model
classifier.to(device)

print(f"Starting training on device: {device}")

epochs = 10

# Save the loss history, loss function CrossEntropyLoss
train_loss = [] # based on training data
val_loss = [] # based on validation data

train_acc_history = []
val_acc_history = [] 

# save best model based on minimum validation loss
best_loss = float("inf")

time_per_epoch = []

for epoch in range(epochs):

    # measure time 
    start_time = time.time()

    print("\n")
    print(f"Epoch {epoch+1}/{epochs}")
    print("-" * 10)

    # TRAINING #
    # Set the model to training mode
    classifier.train()

    num_batches = len(train_loader)
    running_train_loss = 0.0
    correct_train = 0
    total_train = 0

    # looping over batches
    for batch_idx, data in enumerate(train_loader, start=0):

        # get the inputs and labels for each batch
        images, labels = data
        images, labels = images.to(device), labels.to(device)

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = classifier(images)
        loss = criterion(outputs, labels) # For CrossEntropyLoss
        
        loss.backward()
        optimizer.step()


        if batch_idx % 10 == 0:
            print(
                f"Batch {batch_idx}/{num_batches} loss: {loss.item():.4f}"
            )

        # Save the loss for this batch
        running_train_loss += loss.item()
        
        # Calculate training accuracy
        _, predicted = torch.max(outputs, 1)
        correct_train += (predicted == labels).sum().item()
        total_train += labels.size(0)

    # Save the loss and accuracy for this epoch
    avg_train_loss = running_train_loss / num_batches
    train_accuracy = correct_train / total_train

    train_loss.append(avg_train_loss)
    train_acc_history.append(train_accuracy)

    # Print the loss for this epoch
    print("-" * 10)
    print(f"Epoch {epoch+1}/{epochs} : Training loss: {train_loss[-1]:.4f}")
    print(f"Epoch {epoch+1}/{epochs} : Training Accuracy: {train_acc_history[-1]:.4f}")

    # VALIDRITON OF EACH EPOCH #
    # Set the model to evaluation mode
    classifier.eval()
    
    num_batches = len(val_loader)
    running_val_loss = 0.0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for batch_idx, data in enumerate(val_loader, start=0):
        
            images, labels = data
            images, labels = images.to(device), labels.to(device)
            outputs = classifier(images)
            loss = criterion(outputs, labels)  # For CrossEntropyLoss

            # save the loss for this batch
            running_val_loss += loss.item()
            
            # Calculate validation accuracy
            _, predicted = torch.max(outputs, 1)
            correct_val += (predicted == labels).sum().item()
            total_val += labels.size(0)

        # Save the loss for this epoch
        avg_val_loss = running_val_loss / num_batches
        val_accuracy = correct_val / total_val

        val_loss.append(avg_val_loss)
        val_acc_history.append(val_accuracy)

        # Print the loss for this epoch
        print(f"Epoch {epoch+1}/{epochs} : Validation loss: {val_loss[-1]:.4f}")
        print(f"Epoch {epoch+1}/{epochs} : Validation accuracy: {val_acc_history[-1]:.4f}")
        
    # saving the best model

    if avg_val_loss < best_loss:
        best_loss = avg_val_loss
        best_epoch = epoch +1
        torch.save(classifier.state_dict(), os.path.join(output_folder, "best_model.pth"))
        print(f"Best model has been updated...")
     
    # end of epoch time    
    end_time = time.time() 
    elapsed = end_time - start_time
    time_per_epoch.append(elapsed)


# Save the model
torch.save(classifier.state_dict(), os.path.join(output_folder, "last_model.pth"))
print(f"Training completed. Model saved!")
print(f"Best EPOCH: {best_epoch}")

# Save the log
results = ({"epoch": list(range(1, epochs + 1)),
            "time": time_per_epoch,
            "train_accuracy": train_acc_history,
            "val_accuracy": val_acc_history,
            "train_loss": train_loss, 
            "val_loss": val_loss})

results_df = pd.DataFrame(results)
print(results_df.head())

table_save_name = 'training_log'
table_save_name = f'{table_save_name}.csv'
table_save_path = os.path.join(output_folder, table_save_name)
results_df.to_csv(table_save_path, index=False)

### VISUALIZATION OF THE TRAINING PROCESS ###
plt.figure(figsize=(12, 12))

# Training loss
plt.subplot(221)
plt.plot(train_loss, label="Training loss")
plt.title("Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

# Training accuracy
plt.subplot(222)
plt.plot(train_acc_history, label="Training accuracy")
plt.title("Training Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

# Validation loss
plt.subplot(223)
plt.plot(val_loss, label="Validation loss")
plt.title("Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

# Validation accuracy
plt.subplot(224)
plt.plot(val_acc_history, label="Validation accuracy")
plt.title("Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
## load previously trained model
model_path = ""
base_model = models.vit_b_16(weights=None)

# image preprocessing —> need to match training preprocessing
vit_weights = models.ViT_B_16_Weights.IMAGENET1K_V1
preprocess = vit_weights.transforms()

# modify the classifier head to match the number of classes (uses classes from test dataset)
### load classes names from training set ### 
train_dataset = ImageFolder(train_dir, transform=preprocess)
class_names = train_dataset.classes  # 17 jellyfish species
idx_to_class = {i: name for i, name in enumerate(class_names)}
num_classes = len(class_names)
    
# modify the head
base_model.heads.head = nn.Linear(base_model.heads.head.in_features, num_classes)

# load the trained weights
state = torch.load(model_path, map_location=device)
base_model.load_state_dict(state)

In [ ]:
### TEST ###

save = True

classifier = base_model
classifier.to(device)
classifier.eval()

# test data
class_names = test.classes  # 17 jellyfish species
samples = test.samples  # list of (path, class_idx)
results = []  # will hold [filename, true_label, pred_label, confidence]

with torch.no_grad():
    for batch_idx, data in enumerate(test_loader, start=0): #### THIS SHOULD BE test_loader
        
        images, labels = data
        images, labels = images.to(device), labels.to(device)
        
        outputs = classifier(images)  # raw logits
        probs = torch.softmax(outputs, dim=1)           # probabilities
        confs, preds = probs.max(dim=1)                 # best class + confidence

        # compute start index of this batch in the dataset
        start_idx = batch_idx * test_loader.batch_size
        batch_size = images.size(0)
        batch_indices = list(range(start_idx, start_idx + batch_size))

        for idx, true_label_tensor, pred_tensor, conf_tensor in zip(
            batch_indices, labels, preds, confs
        ):
            path, _ = samples[idx]
            filename = os.path.basename(path)  # just filename
            results.append({
                "image": filename,
                "true_label": class_names[true_label_tensor.item()],
                "pred_label": class_names[pred_tensor.item()],
                "confidence": float(conf_tensor.item())
            })


# Create a DataFrame
results_df = pd.DataFrame(results)

if save: 
    table_save_name = 'model_pred'
    table_save_name = f'{table_save_name}.csv'
    table_save_path = os.path.join(output_folder, table_save_name)
    results_df.to_csv(table_save_path, index=False)
    
else:
    print(results_df.head())

def jelly_conf_matrix(data, class_names, class_to_idx, conf_thresh=0.9, save = False):
    """
    Display two confusion matrices side by side:
        Left: counts
        Right: normalized percentages (rounded to 3 digits)
    Args:
        data: pandas DataFrame with ['true_label','pred_label','confidence']
        class_names: list of class names
        class_to_idx: dict mapping class name to numeric index
        conf_thresh: float, filter predictions below this confidence
    """
    # Filter by confidence
    filtered_data = data[data['confidence'] > conf_thresh]

    # Map labels to numeric indices
    y_true = [class_to_idx[label] for label in filtered_data["true_label"]]
    y_pred = [class_to_idx[label] for label in filtered_data["pred_label"]]

    # Compute confusion matrix
    cm_counts = confusion_matrix(y_true, y_pred)
    # Transpose to have x-axis = true labels, y-axis = predicted labels
    cm_counts = cm_counts.T
    
    if save:
        # Save confusion matrix
        plt.figure(figsize=(12, 12))
        disp = ConfusionMatrixDisplay(cm_counts, display_labels=class_names)
        disp.plot(cmap="Blues", xticks_rotation=90, ax=plt.gca())
        plt.xlabel("True Label")
        plt.ylabel("Predicted Label")
        plt.title(f"Confusion Matrix (Counts), Confidence > {conf_thresh}")
        plt.tight_layout()
        plt.savefig(os.path.join(output_folder, f"conf_matrix_counts.png"), dpi = 300, bbox_inches = "tight")
        plt.close()

    else:
        # Display confusion matrix
        plt.figure(figsize=(8, 8))
        disp = ConfusionMatrixDisplay(cm_counts, display_labels=class_names)
        disp.plot(cmap="Blues", xticks_rotation=90, ax=plt.gca())
        plt.xlabel("True Label")
        plt.ylabel("Predicted Label")
        plt.title(f"Confusion Matrix (Counts), Confidence > {conf_thresh}")
        plt.tight_layout()
        plt.show()

def jelly_conf_matrix_normalized(data, class_names, class_to_idx, conf_thresh=0.9, save = False):
    """
    Display a confusion matrix normalized by the number of images per true class.

    Args:
        data: pandas DataFrame with columns ['true_label','pred_label','confidence']
        class_names: list of class names
        class_to_idx: dict mapping class names to numeric indices
        conf_thresh: float, filter predictions below this confidence
    """
    # Filter by confidence
    filtered_data = data[data['confidence'] > conf_thresh]

    # Map labels to numeric indices
    y_true = [class_to_idx[label] for label in filtered_data["true_label"]]
    y_pred = [class_to_idx[label] for label in filtered_data["pred_label"]]

    # Compute confusion matrix
    cm = confusion_matrix(y_true, y_pred)

    # Normalize **row-wise** to account for number of images per true class
    cm_normalized = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

    # Replace NaNs (for classes with 0 samples) with 0
    cm_normalized = np.nan_to_num(cm_normalized, nan=0.0)

    # Transpose so x-axis = true labels, y-axis = predicted labels
    cm_normalized = cm_normalized.T
    
    if save:
        # Save confusion matrix
        plt.figure(figsize=(12, 12))
        disp = ConfusionMatrixDisplay(cm_normalized, display_labels=class_names)
        disp.plot(cmap="Blues", xticks_rotation=90, ax=plt.gca(), values_format=".0f")
        plt.xlabel("True Label")
        plt.ylabel("Predicted Label")
        plt.title(f"Confusion Matrix (Normalized by True Class, %), Confidence > {conf_thresh}")
        plt.tight_layout()
        plt.savefig(os.path.join(output_folder, f"conf_matrix_normalized.png"), dpi = 300, bbox_inches = "tight")
        plt.close()

    else: 
        # Display
        plt.figure(figsize=(8, 8))
        disp = ConfusionMatrixDisplay(cm_normalized, display_labels=class_names)
        disp.plot(cmap="Blues", xticks_rotation=90, ax=plt.gca(), values_format=".0f")
        plt.xlabel("True Label")
        plt.ylabel("Predicted Label")
        plt.title(f"Confusion Matrix (Normalized by True Class, %), Confidence > {conf_thresh}")
        plt.tight_layout()
        plt.show()

jelly_conf_matrix(
    results_df, 
    class_names=test.classes, 
    class_to_idx=test.class_to_idx,
    conf_thresh=0, 
    save = save)

jelly_conf_matrix_normalized(
    results_df, 
    class_names=test.classes, 
    class_to_idx=test.class_to_idx,
    conf_thresh=0, 
    save = save)